In [1]:
import yfinance as yf
import pandas as pd
import re
import requests
import os
from selenium.common.exceptions import NoSuchElementException

import chromedriver_autoinstaller
chromedriver_autoinstaller.install()

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import random

## 나스닥100 티커 리스트

In [2]:
nasdaq100_tickers = ['NVDA','MSFT','AAPL','AMZN','META','AVGO', 'GOOGL','GOOG','TSLA','NFLX','COST','PLTR','ASML','TMUS','CSCO','AMD','AZN','LIN',
 'APP','PEP','SHOP','INTU','PDD','BKNG','MU','QCOM','TXN','ISRG','ARM','AMGN','ADBE','LRCX','GILD','HON','AMAT','PANW','KLAC','CMCSA','ADI','ADP',
 'MELI','INTC','DASH','CRWD','VRTX','CEG','MSTR','CDNS','SBUX','ORLY','CTAS','MDLZ','SNPS','TRI','ABNB','MAR','ADSK','PYPL','MNST','FTNT','CSX',
 'WDAY','AXON','REGN','AEP','MRVL','NXPI','ROP','FAST','PCAR','IDXX','PAYX','ROST','DDOG','CPRT','WBD','TEAM','BKR','TTWO','ZS','EXC','XEL','EA',
 'CCEP','FANG','KDP','CSGP','VRSK','CHTR','MCHP','GEHC','CTSH','KHC','ODFL','DXCM','TTD','CDW','BIIB','ON','LULU','GFS']

In [3]:
len(nasdaq100_tickers)

101

## 전체 티커 순회하면서 추출하는 최종 코드

In [4]:
# -------------------------------
# 1. 뉴스 링크 가져오기
# -------------------------------
def get_href(query, count=20):
    url = f"https://query1.finance.yahoo.com/v1/finance/search?q={query}&newsCount={count}&start=0"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/127.0.0.0 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://finance.yahoo.com/",
        "Connection": "keep-alive",
    }

    response = requests.get(url, headers=headers)
    print(f"[{query}] Status Code: {response.status_code}")

    results = []
    if response.status_code == 200:
        data = response.json()
        for item in data.get("news", []):
            link = item.get("link")
            if link and link.startswith("https://finance.yahoo.com/news/"):
                results.append(link)

    return results[:count]

In [5]:
# -------------------------------
# 2. CSV 업데이트 함수 (PREMIUM 제외)
# -------------------------------
def update_csv(query, csv_path="yahoo_links.csv", premium_path="yahoo_premium.csv"):
    new_links = get_href(query)

    # 기존 CSV 불러오기
    if os.path.exists(csv_path):
        df_old = pd.read_csv(csv_path)
        # old_links = set(df_old["link"])
        old_pairs = set(zip(df_old["ticker"], df_old["link"]))
    else:
        df_old = pd.DataFrame(columns=["ticker", "link", "headline", "pubdate", "related_tickers", "article"])
        #old_links = set()
        old_pairs=set()

    # premium.csv 불러오기
    if os.path.exists(premium_path):
        df_premium = pd.read_csv(premium_path)
        #premium_links = set(df_premium["link"])
        premium_pairs = set(zip(df_premium["ticker"], df_premium["link"]))
    else:
        #premium_links = set()
        premium_pairs = set()
        
    '''
        # 기존에도 없고 premium에도 없는 것만 필터링
        unique_links = [
            link for link in new_links
            if link not in old_links and link not in premium_links
        ]
    '''

    # pairs로 검사하는 것으로 수정
    unique_links = [
        link for link in new_links
        if (query, link) not in old_pairs and (query, link) not in premium_pairs
    ]

    # 새로운 데이터프레임 생성 (티커 포함)
    df_new = pd.DataFrame(unique_links, columns=["link"])
    df_new["ticker"] = query

    # 합치기
    df_updated = pd.concat([df_old, df_new], ignore_index=True)

    # 중복 제거
    df_updated = df_updated.drop_duplicates(subset=["ticker","link"], keep="first")

    # 저장
    df_updated.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"[{query}] 새로운 링크 {len(unique_links)}개 추가됨. 전체 {len(df_updated)}개 저장됨.")
    return df_updated

In [6]:
# -------------------------------
# 3. premium 으로 인해 스킵한 링크 저장하는 함수
# -------------------------------
def save_premium(query, link, csv_path="yahoo_premium.csv"):
    # 기존 파일 불러오기
    if os.path.exists(csv_path):
        df_premium = pd.read_csv(csv_path)
    else:
        df_premium = pd.DataFrame(columns=["ticker", "link"])

    # 새로운 행 추가
    new_row = pd.DataFrame([{"ticker": query, "link": link}])
    df_premium = pd.concat([df_premium, new_row], ignore_index=True)

    # 중복 제거
    df_premium = df_premium.drop_duplicates(subset=["ticker", "link"], keep="first")

    # 저장
    df_premium.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"PREMIUM 저장됨: {link}")

In [7]:
# -------------------------------
# 4. 기사 크롤링 함수
# -------------------------------
def scrape_articles(df, query, csv_path="yahoo_links.csv"):
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    driver = webdriver.Chrome(options=chrome_options)

    i = 0
    while i < len(df):
        link = df.iloc[i]["link"]

        # 이미 headline 있으면 skip
        if "headline" in df.columns and pd.notnull(df.iloc[i]["headline"]):
            i += 1
            continue

        driver.get(link)
        time.sleep(2)

        try:
            # PREMIUM 체크
            head_str = driver.find_element(By.XPATH, '//*[@id="main-content-wrapper"]')
            is_premium = head_str.text.split('\n', 1)[0].strip()
            if is_premium == "PREMIUM":
                print(f"Skip PREMIUM article: {link}")
                save_premium(query, link)   # ⬅️ PREMIUM 따로 저장
                df = df.drop(df.index[i]).reset_index(drop=True)
                continue  # i 증가 안 하고 다음 행으로 (drop 했으니 이미 다음 행이 i 위치에 옴)

            # 기사 내용 추출
            headline = driver.find_element(By.CLASS_NAME, 'cover-headline.yf-1rjrr1').text
            pubdate = driver.find_element(By.CLASS_NAME, 'byline-attr-meta-time').text
            # 관련 티커 (없으면 None)
            try:
                tickers = driver.find_element(By.CLASS_NAME, 'carousel-top').text
            except NoSuchElementException:
                tickers = None
            article = driver.find_element(By.CLASS_NAME, 'bodyItems-wrapper').text

            # "더보기" 버튼 클릭 후 추가 내용
            try:
                continue_button = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Story Continues']")
                continue_button.click()
                time.sleep(1)
                add_article = driver.find_element(By.CLASS_NAME, 'read-more-wrapper').text
                article += "\n" + add_article
            except NoSuchElementException:
                pass

            # DataFrame 업데이트
            df.loc[i, "ticker"] = query
            df.loc[i, "headline"] = headline
            df.loc[i, "pubdate"] = pubdate
            df.loc[i, "related_tickers"] = tickers
            df.loc[i, "article"] = article

            # 정상 처리됐으면 i 증가
            i += 1

        except Exception as e:
            print(f"Error on {link}: {e}")
            # 에러 난 행도 삭제 (다시 시도하지 않음)
            df = df.drop(df.index[i]).reset_index(drop=True)
            # i 증가 안 함 → drop했으니 다음 행이 자동으로 i 위치에 옴

    # 저장 후 종료
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    driver.quit()
    print(f"[{query}] 크롤링 완료 후 저장됨.")


In [8]:
def cleanup_csv(csv_path="yahoo_links.csv"):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        mask = df[["headline", "pubdate", "article"]].isnull().all(axis=1)
        df = df.drop(df.index[mask]).reset_index(drop=True)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"Null 행 {mask.sum()}개 삭제 완료")

In [ ]:
# -------------------------------
# 5. NASDAQ100 전체 순회 실행
# -------------------------------
# 실행 전 클린업 (링크만 있고 기사 데이터 없는 행 제거)
csv_path = "yahoo_links.csv"

cleanup_csv(csv_path)

for ticker in nasdaq100_tickers:
    df = update_csv(ticker, csv_path=csv_path, premium_path="yahoo_premium.csv")
    scrape_articles(df, ticker, csv_path=csv_path)

Null 행 0개 삭제 완료
[NVDA] Status Code: 200
[NVDA] 새로운 링크 6개 추가됨. 전체 4796개 저장됨.
[NVDA] 크롤링 완료 후 저장됨.
[MSFT] Status Code: 200
[MSFT] 새로운 링크 4개 추가됨. 전체 4800개 저장됨.
[MSFT] 크롤링 완료 후 저장됨.
[AAPL] Status Code: 200
[AAPL] 새로운 링크 2개 추가됨. 전체 4802개 저장됨.


In [10]:
df.shape

(4760, 6)